# RQ2 — GoEmotions Fine-Grained Emotion Analysis
**Paper:** Governing a Memory-Safe Language — ICSE 2027  **Platform:** Google Colab

## Model: `monologg/bert-base-cased-goemotions-original`
- 28 output labels: 27 emotions + neutral
- Uses **sigmoid** (not softmax) — emotions are independent
- A comment can have high scores on multiple emotions simultaneously
- Threshold 0.30 used to flag an emotion as present

## Key emotions for governance research
| Emotion | Signal |
|---|---|
| approval / disapproval | Direct community stance |
| curiosity / confusion | Clarity and interest |
| excitement / optimism | Early enthusiasm |
| annoyance / anger | Frustration and conflict |
| gratitude | Appreciation for proposal quality |
| disappointment | Unmet expectations |


---
## Step 0 — Setup

In [ ]:
import os, torch
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import linregress
import warnings; warnings.filterwarnings('ignore')

# ── CHANGE THESE PATHS ─────────────────────────────────────
INPUT_FILE  = '/content/rfc_comments_full.csv'
OUTPUT_FILE = '/content/rfc_goemotions_scores.csv'
AGG_FILE    = '/content/rfc_goemotions_aggregated.csv'
CHECKPOINT  = '/content/goemotions_checkpoint.csv'
ROBERTA_AGG = '/content/rfc_roberta_aggregated.csv'
# ───────────────────────────────────────────────────────────

BOTS       = {'rfcbot', 'rust-highfive', 'bors', 'bors-servo'}
BATCH_SIZE = 32
MAX_LENGTH = 512
THRESHOLD  = 0.30

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print('Estimated: ~20-30 min for 59k comments')
else:
    print('No GPU — Runtime > Change runtime type > T4 GPU')
    print('Without GPU: ~3-4 hours')


Device : cuda
GPU    : NVIDIA A100-SXM4-40GB
Estimated: ~20-30 min for 59k comments


---
## Step 1 — Load GoEmotions Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

MODEL_NAME = 'monologg/bert-base-cased-goemotions-original'
print(f'Loading: {MODEL_NAME}')
print('First run downloads ~400MB...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

EMOTIONS = [
    'admiration','amusement','anger','annoyance','approval',
    'caring','confusion','curiosity','desire','disappointment',
    'disapproval','disgust','embarrassment','excitement','fear',
    'gratitude','grief','joy','love','nervousness','optimism',
    'pride','realization','relief','remorse','sadness','surprise','neutral'
]
print(f'Loaded. {len(EMOTIONS)} emotion labels.')

# Quick test
test_texts = [
    'This is a great proposal, I fully support it!',
    'This design is completely broken and dangerous.',
    'I am confused about how this interacts with lifetimes.',
    'Very interesting idea, curious to see more details.',
]
print('\nQuick test — top 3 emotions per comment:')
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt',
                       truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits)[0].cpu().numpy()
    top3  = sorted(zip(EMOTIONS, probs), key=lambda x: -x[1])[:3]
    top3_str = '  '.join(f'{e}={s:.2f}' for e,s in top3)
    print(f'  "{text[:50]}"')
    print(f'  -> {top3_str}')


Loading: monologg/bert-base-cased-goemotions-original
First run downloads ~400MB...


config.json:   0%|          | 0.00/1.67k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded. 28 emotion labels.

Quick test — top 3 emotions per comment:
  "This is a great proposal, I fully support it!"
  -> admiration=1.00  approval=0.15  excitement=0.00
  "This design is completely broken and dangerous."
  -> fear=0.97  sadness=0.14  neutral=0.12
  "I am confused about how this interacts with lifeti"
  -> confusion=0.99  neutral=0.02  surprise=0.00
  "Very interesting idea, curious to see more details"
  -> curiosity=0.67  excitement=0.64  admiration=0.33


In [ ]:
import pandas as pd

INPUT_FILE = '/content/rfc_comments_full.csv'

# Try three loading strategies in order
try:
    df = pd.read_csv(INPUT_FILE)
    print(f'Standard load: {len(df):,} rows')
except:
    try:
        df = pd.read_csv(INPUT_FILE, engine='python',
                         on_bad_lines='skip')
        print(f'Python engine (bad lines skipped): {len(df):,} rows')
    except:
        df = pd.read_csv(INPUT_FILE, engine='python',
                         quoting=3, on_bad_lines='skip')
        print(f'QUOTE_NONE load: {len(df):,} rows')

print(df['outcome'].value_counts().to_string())

Standard load: 59,771 rows
outcome
accepted     31251
rejected     23437
postponed     5083


---
## Step 2 — Load Comment Data

In [ ]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f'Upload rfc_comments_full.csv to /content/')

df = pd.read_csv(INPUT_FILE)
print(f'Loaded: {len(df):,} rows')
df_human = df[~df['comment_author'].isin(BOTS)].copy()
df_human['comment_body'] = df_human['comment_body'].fillna('').astype(str)
print(f'After bot removal: {len(df_human):,} comments')
print(df_human['outcome'].value_counts().to_string())

# Resume from checkpoint
if os.path.exists(CHECKPOINT):
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['comment_id'].astype(str))
    df_todo  = df_human[~df_human['comment_id'].astype(str).isin(done_ids)].copy()
    print(f'Checkpoint: {len(done_df):,} done, {len(df_todo):,} remaining')
else:
    done_df = pd.DataFrame()
    df_todo = df_human.copy()
    print(f'No checkpoint — scoring all {len(df_todo):,}')


Loaded: 59,771 rows
After bot removal: 58,103 comments
outcome
accepted     30193
rejected     23035
postponed     4875
No checkpoint — scoring all 58,103


---
## Step 3 — Score with GoEmotions

**Note:** GoEmotions uses sigmoid, not softmax.
Each emotion is scored independently 0-1.
Multiple emotions can be high on the same comment.


In [ ]:
import time

def score_batch(texts):
    clean = [t if str(t).strip() else 'okay' for t in texts]
    inputs = tokenizer(clean, return_tensors='pt',
                       truncation=True, max_length=MAX_LENGTH,
                       padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()
    results = []
    for p in probs:
        row = {f'emo_{e}': round(float(p[i]), 4)
               for i,e in enumerate(EMOTIONS)}
        top_idx = p.argmax()
        row['top_emotion'] = EMOTIONS[top_idx]
        row['top_score']   = round(float(p[top_idx]), 4)
        present = [e for i,e in enumerate(EMOTIONS) if p[i] >= THRESHOLD]
        row['emotions_present'] = '|'.join(present) if present else 'none'
        results.append(row)
    return results

texts      = df_todo['comment_body'].tolist()
n          = len(texts)
all_scores = []
start      = time.time()
print(f'Scoring {n:,} comments in batches of {BATCH_SIZE}...')

for i in range(0, n, BATCH_SIZE):
    batch   = texts[i:i+BATCH_SIZE]
    results = score_batch(batch)
    all_scores.extend(results)
    done    = min(i + BATCH_SIZE, n)
    elapsed = time.time() - start
    rate    = done / elapsed if elapsed > 0 else 0
    eta_m   = (n - done) / rate / 60 if rate > 0 else 0
    if done % 5000 < BATCH_SIZE or done == n:
        partial = pd.concat([done_df,
            pd.concat([df_todo.iloc[:done].reset_index(drop=True),
                       pd.DataFrame(all_scores)], axis=1)],
            ignore_index=True)
        partial.to_csv(CHECKPOINT, index=False)
    print(f'  {done:>7}/{n} ({100*done/n:>5.1f}%)  '
          f'{elapsed/60:>5.1f}min  ETA: {eta_m:.1f}min', end='\r')

elapsed = time.time() - start
print(f'\nDone in {elapsed/60:.1f} minutes')
new_df    = pd.concat([df_todo.reset_index(drop=True),
                       pd.DataFrame(all_scores)], axis=1)
df_scored = pd.concat([done_df, new_df], ignore_index=True)
print(f'Total scored: {len(df_scored):,}')
print('\nMost common top emotions:')
print(df_scored['top_emotion'].value_counts().head(10).to_string())


Scoring 58,103 comments in batches of 32...
    58103/58103 (100.0%)    6.0min  ETA: 0.0min
Done in 6.0 minutes
Total scored: 58,103

Most common top emotions:
top_emotion
neutral        33981
confusion       5365
approval        4018
curiosity       3584
gratitude       2474
disapproval     1408
optimism        1366
admiration      1215
realization     1110
annoyance        635


---
## Step 4 — Aggregate Per RFC

In [ ]:
def aggregate_rfc(group):
    g = group.sort_values('comment_order')
    n = len(g)
    result = {'n_comments': n}
    for e in EMOTIONS:
        col = f'emo_{e}'
        if col in g.columns:
            result[f'mean_{e}'] = round(float(g[col].mean()), 4)
            result[f'rate_{e}'] = round(float((g[col] >= THRESHOLD).mean()), 4)
    apr = result.get('rate_approval', 0)
    dis = result.get('rate_disapproval', 0)
    denom = apr + dis
    result['goe_ADR'] = round(apr / denom, 4) if denom > 0 else np.nan
    if 'emo_approval' in g.columns and 'emo_disapproval' in g.columns:
        net = (g['emo_approval'] - g['emo_disapproval']).values
        result['mean_net_approval'] = round(float(net.mean()), 4)
        if n >= 3:
            orders = g['comment_order'].values
            slope, _, r, p, _ = linregress(orders, net)
            result['goe_beta_s']  = round(float(slope), 6)
            result['goe_beta_r2'] = round(float(r**2), 4)
            result['goe_beta_p']  = round(float(p), 4)
        else:
            result['goe_beta_s'] = result['goe_beta_r2'] = result['goe_beta_p'] = np.nan
    if 'top_emotion' in g.columns:
        result['dominant_emotion'] = g['top_emotion'].value_counts().index[0]
    return pd.Series(result)

print('Aggregating per RFC...')
agg = df_scored.groupby(['pr_number','rfc_number','outcome']).apply(
    aggregate_rfc).reset_index()
print(f'Done. {len(agg)} RFCs, {len(agg.columns)} columns')
key = [c for c in ['pr_number','outcome','n_comments',
       'mean_approval','rate_approval','mean_disapproval',
       'rate_disapproval','mean_curiosity','mean_confusion',
       'mean_excitement','goe_ADR','dominant_emotion'] if c in agg.columns]
print(agg[key].head(6).to_string())


Aggregating per RFC...
Done. 1525 RFCs, 66 columns
   pr_number   outcome  n_comments  mean_approval  rate_approval  mean_disapproval  rate_disapproval  mean_curiosity  mean_confusion  mean_excitement  goe_ADR dominant_emotion
0          1  accepted           8         0.1229         0.1250            0.0008            0.0000          0.1186          0.2197           0.0004      1.0          neutral
1          2  accepted          10         0.0798         0.1000            0.0019            0.0000          0.0036          0.0048           0.0076      1.0          neutral
2          3  accepted          22         0.1845         0.1818            0.0145            0.0000          0.1166          0.1090           0.0006      1.0          neutral
3          4  rejected           9         0.0010         0.0000            0.0378            0.1111          0.1221          0.1938           0.0013      0.0          neutral
4          5  rejected          12         0.0029         0.0000     

---
## Step 5 — Compare Key Emotions Across Outcomes

In [ ]:
outcomes = ['accepted','rejected','postponed']
gov_emotions = [
    ('approval','Approval'), ('disapproval','Disapproval'),
    ('curiosity','Curiosity'), ('confusion','Confusion'),
    ('excitement','Excitement'), ('annoyance','Annoyance'),
    ('anger','Anger'), ('gratitude','Gratitude'),
    ('optimism','Optimism'), ('disappointment','Disappointment'),
    ('admiration','Admiration'), ('neutral','Neutral'),
]
print('GOEMOTIONS MEAN SCORE BY OUTCOME')
print(f'{"Emotion":<16}', end='')
for o in outcomes: print(f'{o:>12}', end='')
print(f'{"p":>12} {"sig":>5} {"r":>7}')
print('-'*65)
res = []
for emo, label in gov_emotions:
    col = f'mean_{emo}'
    if col not in agg.columns: continue
    groups = {o: agg[agg['outcome']==o][col].dropna() for o in outcomes}
    a, r_g = groups.get('accepted',pd.Series()), groups.get('rejected',pd.Series())
    if len(a)>=3 and len(r_g)>=3:
        u, p = stats.mannwhitneyu(a, r_g, alternative='two-sided')
        eff = abs(1-(2*u)/(len(a)*len(r_g)))
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    else:
        p = eff = np.nan; sig = ''
    print(f'{label:<16}', end='')
    for o in outcomes:
        m = groups[o].median() if len(groups.get(o,[])) > 0 else np.nan
        print(f'{m:>12.4f}', end='')
    print(f'{p:>12.4f} {sig:>5} {eff:>7.3f}')
    res.append({'emotion':emo,'acc':a.median(),'rej':r_g.median(),'p':p,'r':eff,'sig':sig})
res_df = pd.DataFrame(res)
print('\nSignificant findings ranked by effect size:')
print(res_df[res_df['sig'].isin(['*','**','***'])].sort_values('r',ascending=False).to_string())


GOEMOTIONS MEAN SCORE BY OUTCOME
Emotion             accepted    rejected   postponed           p   sig       r
-----------------------------------------------------------------
Approval              0.0988      0.0832      0.0842      0.0000   ***   0.142
Disapproval           0.0163      0.0194      0.0201      0.0092    **   0.080
Curiosity             0.0919      0.0774      0.0964      0.0015    **   0.098
Confusion             0.1097      0.1167      0.1355      0.2408    ns   0.036
Excitement            0.0011      0.0007      0.0010      0.0000   ***   0.278
Annoyance             0.0054      0.0046      0.0049      0.3478    ns   0.029
Anger                 0.0007      0.0007      0.0007      0.2066    ns   0.039
Gratitude             0.0430      0.0306      0.0389      0.0112     *   0.078
Optimism              0.0328      0.0214      0.0433      0.0000   ***   0.137
Disappointment        0.0029      0.0027      0.0051      0.7987    ns   0.008
Admiration            0.0226    

---
## Step 6 — GoEmotions ADR

In [ ]:
for col, label in [
    ('goe_ADR', 'GoEmotions ADR — approval/(approval+disapproval)'),
    ('mean_net_approval', 'Net approval score (approval - disapproval)'),
]:
    if col not in agg.columns: continue
    print(f'\n{label}')
    print(f'{"Stat":<12}', end='')
    for o in outcomes: print(f'{o:>13}', end='')
    print()
    print('-'*51)
    groups = {o: agg[agg['outcome']==o][col].dropna() for o in outcomes}
    for stat, fn in [('n',lambda s:f'{len(s)}'),('median',lambda s:f'{s.median():.4f}'),
                     ('mean',lambda s:f'{s.mean():.4f}')]:
        print(f'{stat:<12}', end='')
        for o in outcomes: print(f'{fn(groups[o]):>13}', end='')
        print()
    a, r_g = groups['accepted'], groups['rejected']
    if len(a)>=3 and len(r_g)>=3:
        u,p = stats.mannwhitneyu(a, r_g, alternative='two-sided')
        eff = abs(1-(2*u)/(len(a)*len(r_g)))
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        print(f'  MW accepted vs rejected: p={p:.4f} {sig}  r={eff:.3f}')



GoEmotions ADR — approval/(approval+disapproval)
Stat             accepted     rejected    postponed
---------------------------------------------------
n                     566          630           91
median             0.8337       0.7574       0.7999
mean               0.7993       0.7214       0.7569
  MW accepted vs rejected: p=0.0001 ***  r=0.126

Net approval score (approval - disapproval)
Stat             accepted     rejected    postponed
---------------------------------------------------
n                     624          799          102
median             0.0696       0.0465       0.0519
mean               0.0836       0.0552       0.0582
  MW accepted vs rejected: p=0.0000 ***  r=0.195


---
## Step 7 — Dominant Emotion by Outcome

In [ ]:
if 'dominant_emotion' in agg.columns:
    for o in outcomes:
        g  = agg[agg['outcome']==o]
        vc = g['dominant_emotion'].value_counts()
        print(f'\n{o} (n={len(g)}):')
        for emo, cnt in vc.head(8).items():
            pct = 100*cnt/len(g)
            print(f'  {emo:<18} {cnt:>5} ({pct:>5.1f}%) {"*"*int(pct/2)}')



accepted (n=624):
  neutral              609 ( 97.6%) ************************************************
  curiosity              3 (  0.5%) 
  approval               3 (  0.5%) 
  confusion              2 (  0.3%) 
  admiration             2 (  0.3%) 
  optimism               2 (  0.3%) 
  surprise               1 (  0.2%) 
  annoyance              1 (  0.2%) 

rejected (n=799):
  neutral              741 ( 92.7%) **********************************************
  confusion             12 (  1.5%) 
  gratitude             11 (  1.4%) 
  approval               8 (  1.0%) 
  disapproval            6 (  0.8%) 
  curiosity              5 (  0.6%) 
  remorse                4 (  0.5%) 
  admiration             3 (  0.4%) 

postponed (n=102):
  neutral              100 ( 98.0%) *************************************************
  admiration             1 (  1.0%) 
  approval               1 (  1.0%) 


---
## Step 8 — GoEmotions ADR vs RoBERTa ADR

In [ ]:
if os.path.exists(ROBERTA_AGG) and 'goe_ADR' in agg.columns:
    rob = pd.read_csv(ROBERTA_AGG)
    merged = agg.merge(rob[['pr_number','outcome','ADR']],
                       on=['pr_number','outcome'])
    common = merged[['goe_ADR','ADR']].dropna()
    rho,p  = stats.spearmanr(common['goe_ADR'], common['ADR'])
    agree  = 'Strong' if abs(rho)>0.7 else 'Moderate' if abs(rho)>0.4 else 'Weak'
    print(f'GoEmotions ADR vs RoBERTa ADR agreement:')
    print(f'  Spearman rho={rho:.3f}  p={p:.4f}  {agree}  n={len(common):,}')
    print(f'\n  GoEmotions ADR by outcome:')
    for o in outcomes:
        m = agg[agg['outcome']==o]['goe_ADR'].dropna().median()
        print(f'    {o:<12}: {m:.4f}')
    print(f'  RoBERTa ADR by outcome:')
    for o in outcomes:
        m = rob[rob['outcome']==o]['ADR'].dropna().median()
        print(f'    {o:<12}: {m:.4f}')
else:
    print('RoBERTa file not found or goe_ADR missing — skipping')


RoBERTa file not found or goe_ADR missing — skipping


---
## Step 9 — Save and Download

In [ ]:
from google.colab import files as colab_files

base_cols = ['pr_number','rfc_number','outcome','comment_id',
             'comment_author','comment_order','comment_created_at',
             'top_emotion','top_score','emotions_present']
emo_cols  = [f'emo_{e}' for e in EMOTIONS]
save_cols = [c for c in base_cols+emo_cols if c in df_scored.columns]
df_scored[save_cols].to_csv(OUTPUT_FILE, index=False)
print(f'Saved: {OUTPUT_FILE}  ({len(df_scored):,} rows)')
agg.to_csv(AGG_FILE, index=False)
print(f'Saved: {AGG_FILE}  ({len(agg):,} rows)')
if os.path.exists(CHECKPOINT): os.remove(CHECKPOINT)
print('\nDownloading...')
colab_files.download(OUTPUT_FILE)
colab_files.download(AGG_FILE)
print('Done — check browser downloads')


Saved: /content/rfc_goemotions_scores.csv  (58,103 rows)
Saved: /content/rfc_goemotions_aggregated.csv  (1,525 rows)

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done — check browser downloads
